# 17 · Final test, once, then stress_bear (E17)

Step 6 of the initiation plan. Everything was chosen on train and validation in E14-E16: the factor model (E15:
re-tuned GRU, 3 seeds averaged), the agent's settings (E16 tuning), each seed's checkpoint, and each seed's
stay-flat decision (none fell back). Nothing below is chosen or tuned. The **5 saved E16 agents** (Double DQN, state A:
market factors only) and the baselines are backtested **once** on:

* **test**, 2025-07-01 → 2025-10-30: the held-out period, reported as the result;
* **stress_bear**, 2025-11-01 → 2026-06-06: a falling market (buy and hold about −44 %), reported separately as
  a robustness check.

Same costs and execution as before (0.10 % fee + 0.02 % slippage per side, orders at the next bar, a decision every
5 minutes). A policy whose equity reaches 0 counts as ruined (−100 %, Sharpe undefined): the environment has no
margin call.

**Run once.** Results are written to `thesis_rl/initiation/e17_final/`. A second run of this notebook only shows the
saved results; it does not backtest again. A GPU is not needed: a CPU runtime is enough (a few minutes in total).

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/14-initiation"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
seeding.record_environment(L.report / "environment", "17_initiation_final_test")

In [ ]:
import json, time, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from rltrade.runner import backtest_saved, RunSpec
from rltrade.env import EnvConfig
from rltrade import results as R

INIT = L.root / "initiation"
AG, REP, FINAL = INIT / "agents", INIT / "report", INIT / "e17_final"
for p in (REP, FINAL): p.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/initiation_prepared") if "google.colab" in sys.modules else INIT / "prepared"
ZIP = INIT / "initiation_prepared.zip"
PREP = LOCAL / "fixed_plus"
SPLITS = ("test", "stress_bear")
NEEDED = [f"fixed_plus/market_{s}.npz" for s in SPLITS] + ["fixed_plus/meta.json"]
if not all((LOCAL / n).exists() for n in NEEDED):
    assert ZIP.exists(), f"upload initiation_prepared.zip to {ZIP}"
    with zipfile.ZipFile(ZIP) as z: z.extractall(LOCAL, members=NEEDED)
    print("unzipped", NEEDED, "->", LOCAL)

sel = json.loads((INIT / "selected_agent_e16.json").read_text())
def on_drive(p):                                    # paths written on another machine: same place under initiation/
    p = Path(p)
    return p if p.exists() else INIT.joinpath(*p.parts[p.parts.index("initiation") + 1:])
F = on_drive(sel["factor_dir"])
BASELINES = [RunSpec("16_agents", b, env=EnvConfig(state="A")).run_id for b in ("buy_and_hold", "always_short", "cash")]
RUNS = list(sel["runs"]) + BASELINES
for r in RUNS:
    assert (AG / "models" / r / "metrics.json").exists(), f"missing E16 run {r}"
for s in SPLITS:
    f, m = np.load(F / f"factors_{s}.npy", mmap_mode="r"), np.load(PREP / f"market_{s}.npz")
    assert f.shape == (len(m["midpoint"]), 8), (s, f.shape)
val = R.load_results(AG).set_index("run_id")
print(f"{len(sel['runs'])} agents ({sel['agent']}, states {sel['states']}) + {len(BASELINES)} baselines | factors {F}")
print("fallback rule:", sel["fallback"])

In [ ]:
KEYS = ("cum_return_pct", "annual_return_pct", "sharpe", "sortino", "max_drawdown_pct", "romad", "win_rate_pct",
        "num_trades", "exposure_pct", "long_pct", "short_pct", "costs_pct", "steps")

def evaluate(split):
    """Backtest every run on `split` once; later calls load the saved results instead."""
    out = FINAL / f"{split}_runs.csv"
    if out.exists():
        info = json.loads((FINAL / f"{split}_info.json").read_text())
        print(f"{split}: already evaluated at {info['time']} (code {info['commit']}) - showing the saved results")
        return pd.read_csv(out)
    rows, curves = [], {}
    for rid in RUNS:
        rd = AG / "models" / rid
        mets, bts = backtest_saved(rd, [split], prepared_dir=PREP, factor_dir=F)
        m, bt = mets[split], bts[split]
        ruined = bool((bt["equity"] <= 0).any())
        if ruined:                                   # no margin call in the environment: equity <= 0 is ruin
            m = dict(m, cum_return_pct=-100.0, annual_return_pct=-100.0, max_drawdown_pct=100.0, sharpe=float("nan"))
        cfg = json.loads((rd / "config.json").read_text())["spec"]
        row0 = json.loads((rd / "metrics.json").read_text())["row"]
        rows.append({"run_id": rid, "agent": cfg["agent"], "seed": cfg["seed"], "fallback_used": row0.get("fallback_used", ""),
                     "ruined": ruined, **{k: m[k] for k in KEYS}})
        curves[rid.split("/")[-1]] = np.clip(bt["equity"][1:], 0, None)
        curves["time_ns"] = bt["time_ns"]
        print(f"  {rid}: done")
    df = pd.DataFrame(rows)
    df.to_csv(out, index=False)
    np.savez_compressed(FINAL / f"{split}_curves.npz", **curves)
    (FINAL / f"{split}_info.json").write_text(json.dumps({"time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "commit": seeding.git_commit(EXP), "runs": RUNS, "factor_dir": str(F)}, indent=1))
    return df

def ms(x, f="+.2f"):
    x = pd.to_numeric(pd.Series(x), errors="coerce").dropna()
    return f"{x.mean():{f}} ± {x.std(ddof=1):.{f[-2]}f}" if len(x) > 1 else (f"{x.mean():{f}}" if len(x) else "n/a")

def summarise(split, df):
    ag, bl = df[df.agent == sel["agent"]].sort_values("seed"), df[df.agent != sel["agent"]]
    bh = float(bl[bl.agent == "buy_and_hold"].sharpe.iloc[0])
    per_seed = ag[["seed", "sharpe", "cum_return_pct", "max_drawdown_pct", "num_trades", "exposure_pct", "long_pct",
                   "short_pct", "costs_pct", "ruined"]].copy()
    per_seed.insert(1, "val_sharpe", [val.loc[r, "val_sharpe"] for r in ag.run_id])
    rows = [{"policy": f"DDQN state A (5 seeds)", "Sharpe": ms(ag.sharpe), "return %": ms(ag.cum_return_pct),
             "max DD %": ms(ag.max_drawdown_pct, ".2f"), "trades": ms(ag.num_trades, ".0f"),
             "beat buy and hold": f"{int((ag.sharpe > bh).sum())} of {len(ag)}"}]
    for _, r in bl.iterrows():
        rows.append({"policy": r.agent, "Sharpe": "ruined" if r.ruined else f"{r.sharpe:+.2f}",
                     "return %": f"{r.cum_return_pct:+.2f}", "max DD %": f"{r.max_drawdown_pct:.2f}",
                     "trades": int(r.num_trades), "beat buy and hold": ""})
    summ = pd.DataFrame(rows)
    t = stats.ttest_1samp(ag.sharpe.dropna(), bh)    # 5 seeds against one buy-and-hold number: indicative only
    note = (f"agents vs buy and hold Sharpe {bh:+.2f}: mean difference {ag.sharpe.mean() - bh:+.2f}, "
            f"one-sample t {t.statistic:+.2f}, p = {t.pvalue:.3f} (n = {ag.sharpe.notna().sum()} seeds)")
    print(f"===== {split}\n" + summ.to_string(index=False) + "\n\n" + note + "\n\nper seed:\n"
          + per_seed.round(3).to_string(index=False))
    return summ, per_seed, note

## 1 · Test (the held-out result, once)

In [ ]:
res = {"test": evaluate("test")}
out = {"test": summarise("test", res["test"])}

## 2 · stress_bear (robustness in a falling market)

In [ ]:
res["stress_bear"] = evaluate("stress_bear")
out["stress_bear"] = summarise("stress_bear", res["stress_bear"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
style = {"buy_and_hold": ("black", "--"), "always_short": ("grey", "--"), "cash": ("lightgrey", "--")}
for ax, s in zip(axes, SPLITS):
    z = np.load(FINAL / f"{s}_curves.npz"); t = R.as_datetime(z["time_ns"])
    for k in z.files:
        if k == "time_ns": continue
        name = k.split("_A_")[0]
        if name in style: ax.plot(t, z[k], lw=1.3, color=style[name][0], ls=style[name][1], label=name)
        else: ax.plot(t, z[k], lw=1, alpha=0.85, label=f"DDQN seed {k.rsplit('_s', 1)[-1]}")
    ax.set_title(f"{s}: equity, net of costs"); ax.grid(alpha=0.3); ax.tick_params(axis="x", rotation=30)
axes[0].set_ylabel("equity (start = 1)"); axes[1].legend(fontsize=7, loc="upper left")
fig.tight_layout(); fig.savefig(REP / "e17_final_equity.png", dpi=130); plt.show()

In [ ]:
txt = ["# E17 final evaluation (single pass; nothing chosen on these splits)\n",
       f"Agents: {sel['agent']}, state {sel['states']}, seeds {sel['seeds']}; factors {Path(sel['factor_dir']).name}; {sel['fallback']}.\n"]
for s in SPLITS:
    summ, per_seed, note = out[s]
    txt += [f"## {s}\n", summ.to_markdown(index=False), "", note, "", "Per seed:", "", per_seed.round(3).to_markdown(index=False), ""]
(REP / "e17_final.md").write_text("\n".join(txt) + "\n", encoding="utf-8")
print("wrote", REP / "e17_final.md", "|", REP / "e17_final_equity.png", "|", FINAL)

In [ ]:
AUTO_DISCONNECT = True               # release the runtime once the results are on Drive
if AUTO_DISCONNECT and "google.colab" in sys.modules:
    from google.colab import runtime
    runtime.unassign()